# Silver layer transformations for ERP Customer data
1. Check data types
2. Trim data
3. Rename column headers
4. Normalize customer ID between ERP and CRM data, normalize gender category
5. Birthdate validation for annomaly dates

## Read data and spark libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

df = spark.table("workspace.bronze.erp_cust_az12")
df.display()

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID cleanup
Leave the AW******* part of the code

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"),4, F.length(col("cid")))) # Take everything after the 4th element
    .otherwise(col("cid"))
)

df.display()

##Birthdate validation

In [0]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
    .otherwise(col("bdate"))
)

## Gender normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

df.display()

## Rename columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write data to silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer")